# Dionaea SQLite FTP Authentication and Credential Analysis

---

## Dataset Overview
* **Source:** Raw Dionaea SQLite databases (`dionaea.sqlite.*`).
* **Target Protocol:** FTP (`ftpd` connections and corresponding login records).
* **Scope:** Relational extraction of connections and authentication attempts to analyze brute-force activity.

---

## Processing Steps
1. **Relational Ingestion:** Connected to SQLite databases and merged `connections` and `logins` tables using `connection` IDs.
2. **Data Sanitization:** Handled missing values, dropped completely empty columns, and resolved byte-encoding issues (`decode_bytes`) in username/password fields.
3. **Temporal Breakdown:** Extracted attack timestamps into granular time features (`hour`, `weekday`).
4. **Behavioral Analysis:** Identified top attacker IP addresses, most targeted credentials, and login session frequencies.

---

## Final Output
* **File:** `dionaea_ftp_clean.csv` – A cleaned relational dataset for analyzing FTP credential targeting and brute-force patterns.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import glob

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [ ]:
db_files = sorted(glob.glob("/content/dionaea.sqlite.*"))

print("Number of SQLite files:", len(db_files))
print(db_files[:5])

Number of SQLite files: 30
['/content/dionaea.sqlite.1', '/content/dionaea.sqlite.10', '/content/dionaea.sqlite.11', '/content/dionaea.sqlite.12', '/content/dionaea.sqlite.13']


In [ ]:
conn = sqlite3.connect(db_files[0])

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)

tables

,name
0,connections
1,dcerpcbinds
2,dcerpcrequests
3,dcerpcserviceops
4,dcerpcservices
5,downloads
6,emu_profiles
7,emu_services
8,logins
9,mqtt_fingerprints


In [ ]:
ftp_frames = []

for db in db_files:

    conn = sqlite3.connect(db)

    df = pd.read_sql_query("""

    SELECT *

    FROM connections

    WHERE connection_protocol='ftpd'

    """, conn)

    ftp_frames.append(df)

    conn.close()

ftp_connections = pd.concat(
    ftp_frames,
    ignore_index=True
)

print(ftp_connections.shape)

ftp_connections.head()

(187, 12)


/tmp/ipykernel_562/2049879044.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ftp_connections = pd.concat(


,connection,connection_type,connection_transport,connection_protocol,connection_timestamp,connection_root,connection_parent,local_host,local_port,remote_host,remote_hostname,remote_port
0,639,accept,tcp,ftpd,1.581746e+09,639,None,167.172.104.173,21,192.241.231.243,,56190
1,20231,accept,tcp,ftpd,1.581785e+09,20231,None,167.172.104.173,21,104.206.128.30,,63509
2,20894,accept,tcp,ftpd,1.581786e+09,20894,None,167.172.104.173,21,184.105.247.252,,30662
3,122,accept,tcp,ftpd,1.581654e+09,122,None,167.172.104.173,21,172.105.218.213,,60492
4,556,accept,tcp,ftpd,1.581659e+09,556,None,167.172.104.173,21,192.241.239.139,,47388


In [ ]:
print("Rows   :", ftp_connections.shape[0])
print("Columns:", ftp_connections.shape[1])

ftp_connections.columns.tolist()

Rows   : 187
Columns: 12


['connection',
 'connection_type',
 'connection_transport',
 'connection_protocol',
 'connection_timestamp',
 'connection_root',
 'connection_parent',
 'local_host',
 'local_port',
 'remote_host',
 'remote_hostname',
 'remote_port']

In [ ]:
print("Duplicate Rows:", ftp_connections.duplicated().sum())

Duplicate Rows: 0


In [ ]:
ftp_connections.dtypes

,0
connection,object
connection_type,object
connection_transport,object
connection_protocol,object
connection_timestamp,float64
connection_root,object
connection_parent,object
local_host,object
local_port,object
remote_host,object


In [ ]:
ftp_connections.isna().sum().sort_values(ascending=False)

,0
connection_parent,187
connection,0
connection_transport,0
connection_type,0
connection_protocol,0
connection_timestamp,0
connection_root,0
local_host,0
local_port,0
remote_host,0


In [ ]:
empty_cols = ftp_connections.columns[
    ftp_connections.isna().all()
]

print(empty_cols)

Index(['connection_parent'], dtype='object')


In [ ]:
ftp_connections = ftp_connections.drop(
    columns=empty_cols
)

ftp_connections.shape

(187, 11)

In [ ]:
ftp_connections["timestamp"] = pd.to_datetime(
    ftp_connections["connection_timestamp"],
    unit="s"
)

ftp_connections[
    ["connection_timestamp","timestamp"]
].head()

,connection_timestamp,timestamp
0,1.581746e+09,2020-02-15 05:48:27.558816671
1,1.581785e+09,2020-02-15 16:36:56.719931841
2,1.581786e+09,2020-02-15 17:03:10.799254656
3,1.581654e+09,2020-02-14 04:21:26.532921314
4,1.581659e+09,2020-02-14 05:43:28.101692438


In [ ]:
print(
    "Start:",
    ftp_connections["timestamp"].min()
)

print(
    "End:",
    ftp_connections["timestamp"].max()
)

Start: 2020-01-22 06:15:12.326608181
End: 2020-02-22 01:34:44.148376703


In [ ]:
ftp_connections["date"] = (
    ftp_connections["timestamp"].dt.date
)

ftp_connections["date"].value_counts().sort_index()

,count
date,
2020-01-22,15
2020-01-23,11
2020-01-24,5
2020-01-25,9
2020-01-26,6
2020-02-02,2
2020-02-03,5
2020-02-04,8
2020-02-05,10


In [ ]:
ftp_connections["connection_protocol"].value_counts()

,count
connection_protocol,
ftpd,187


In [ ]:
ftp_connections["connection_transport"].value_counts()

,count
connection_transport,
tcp,187


In [ ]:
ftp_connections["connection_type"].value_counts()

,count
connection_type,
accept,187


In [ ]:
db_files = sorted(glob.glob("/content/dionaea.sqlite.*"))
print(len(db_files))

30


In [ ]:
login_frames = []

for db in db_files:

    conn = sqlite3.connect(db)

    try:
        df = pd.read_sql_query(
            "SELECT * FROM logins",
            conn
        )
        login_frames.append(df)
    except:
        pass

    conn.close()

ftp_logins = pd.concat(
    login_frames,
    ignore_index=True
)

ftp_logins.shape

(1487377, 4)

In [ ]:
ftp_logins.head()

,login,connection,login_username,login_password
0,1,80,b'root',
1,2,81,b'root',
2,3,84,b'root',
3,4,85,b'root',
4,5,86,b'root',


In [ ]:
ftp_dataset = ftp_connections.merge(
    ftp_logins,
    on="connection",
    how="left"
)

ftp_dataset.shape

(1344, 16)

In [ ]:
ftp_dataset.columns.tolist()

['connection',
 'connection_type',
 'connection_transport',
 'connection_protocol',
 'connection_timestamp',
 'connection_root',
 'local_host',
 'local_port',
 'remote_host',
 'remote_hostname',
 'remote_port',
 'timestamp',
 'date',
 'login',
 'login_username',
 'login_password']

In [ ]:
ftp_dataset[
    ["login_username", "login_password"]
].notna().sum()

,0
login_username,1335
login_password,1335


In [ ]:
ftp_dataset.dtypes

,0
connection,object
connection_type,object
connection_transport,object
connection_protocol,object
connection_timestamp,float64
connection_root,object
local_host,object
local_port,object
remote_host,object
remote_hostname,object


In [ ]:
def decode_bytes(x):
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="ignore")
    return x

ftp_dataset["login_username"] = ftp_dataset["login_username"].apply(decode_bytes)
ftp_dataset["login_password"] = ftp_dataset["login_password"].apply(decode_bytes)

In [ ]:
ftp_dataset[
    ["login_username", "login_password"]
].head(20)

,login_username,login_password
0,sa,dddddddd
1,sa,1qazxsw2
2,sa,0123456789
3,sa,feifei521
4,sa,19890125
5,sa,654321123456
6,sa,11335577
7,sa,14789632
8,sa,woaiwoma
9,sa,19820512


In [ ]:
ftp_dataset[
    ["login_username","login_password"]
].isna().sum()

,0
login_username,9
login_password,9


In [ ]:
ftp_dataset["login_username"]\
.value_counts(dropna=False)\
.head(20)

,count
login_username,
sa,1274
anonymous,22
NaN,9
root,4
PlcmSpIp,3
php,2
mssqla,2
win,1
Administrator,1


In [ ]:
ftp_dataset["login_password"]\
 .value_counts(dropna=False)\
.head(20)

,count
login_password,
,20
anonymous,14
NaN,9
anonymous@,7
qqqqqqqq,4
000000000,3
!@#$%^&*,3
369369369,3
aa123456,3


In [ ]:
print(
    "Unique usernames:",
    ftp_dataset["login_username"].nunique(dropna=True)
)

Unique usernames: 34


In [ ]:
print(
    "Unique passwords:",
    ftp_dataset["login_password"].nunique(dropna=True)
)

Unique passwords: 1197


In [ ]:
credential_pairs = (
    ftp_dataset[
        ["login_username","login_password"]
    ]
    .dropna()
    .value_counts()
)

credential_pairs.head(20)

login_username  login_password
anonymous       anonymous         14
                anonymous@         7
sa              qqqqqqqq           4
root                               4
sa              19810914           3
                1q2w3e4r           3
                369369369          3
                !@#$%^&*           3
                19870905           3
                19861218           3
                11111111           3
                19881020           3
                qazwsxedc          3
                000000000          3
                19890101           3
                521521521          2
                52113140           2
                sql2000            2
                ssssssss           2
                qwertyuiop         2
Name: count, dtype: int64

In [ ]:
ftp_dataset[
    ftp_dataset["login_username"]=="sa"
][
    ["connection","connection_protocol"]
].head(20)

,connection,connection_protocol
0,639,ftpd
1,639,ftpd
2,639,ftpd
3,20231,ftpd
4,20231,ftpd
5,20231,ftpd
6,20231,ftpd
7,20231,ftpd
8,20231,ftpd
9,20231,ftpd


In [ ]:
ftp_dataset.groupby("connection_protocol")[
    "login_username"
].count()

,login_username
connection_protocol,
ftpd,1335


In [ ]:
ftp_dataset.duplicated().sum()

np.int64(0)

print("Total Sessions:", ftp_dataset["connection"].nunique())

In [ ]:
ftp_dataset["connection"].value_counts().head(20)

,count
connection,
9133,13
5606,13
10154,12
7064,12
10100,12
9009,12
13228,12
36421,12
6719,11


In [ ]:
session_attempts = (
    ftp_dataset.groupby("connection")
    .size()
    .sort_values(ascending=False)
)

print(session_attempts.describe())

count    186.000000
mean       7.225806
std        2.970591
min        1.000000
25%        5.000000
50%        8.000000
75%        9.750000
max       13.000000
dtype: float64


In [ ]:
session_attempts.head(20)

,0
connection,
5606,13
9133,13
10154,12
36421,12
13228,12
10100,12
7064,12
9009,12
5658,11


In [ ]:
ftp_dataset["login_username"].isna().groupby(
    ftp_dataset["connection"]
).all().value_counts()

,count
login_username,
False,178
True,8


In [ ]:
login_sessions = ftp_dataset.groupby("connection")[
    "login_username"
].apply(lambda x: x.notna().any())

login_sessions.value_counts()

,count
login_username,
True,178
False,8


In [ ]:
login_attempts = (
    ftp_dataset
    .dropna(subset=["login_username"])
    .groupby("connection")
    .size()
    .sort_values(ascending=False)
)

login_attempts.head(20)

,0
connection,
9133,13
5606,13
36421,12
13228,12
10100,12
9009,12
10154,12
7064,12
8574,11


In [ ]:
ftp_dataset[
    ftp_dataset["login_username"].isna()
]["connection"].nunique()

8

In [ ]:
print("Unique attacker IPs:", ftp_dataset["remote_host"].nunique())

Unique attacker IPs: 104


In [ ]:
top_ips = ftp_dataset["remote_host"].value_counts()

top_ips.head(20)

,count
remote_host,
51.15.151.235,116
83.97.20.34,60
35.233.105.134,50
172.105.218.213,46
164.52.24.165,42
164.68.112.178,36
184.105.247.194,31
35.195.54.74,26
35.205.86.202,25


In [ ]:
top_ips.describe()

,count
count,104.000000
mean,12.923077
std,14.553345
min,1.000000
25%,6.000000
50%,9.000000
75%,15.000000
max,116.000000


In [ ]:
(top_ips == 1).sum()

np.int64(6)

In [ ]:
(top_ips > 1).sum()

np.int64(98)

In [ ]:
ftp_dataset.groupby("login_username")["login_password"]\
    .nunique()\
    .sort_values(ascending=False)\
    .head(20)

,login_password
login_username,
sa,1176
PlcmSpIp,3
anonymous,3
mssqla,2
Administrator,1
0004F21FDD08,1
Poly310PB!dP,1
PolyVVX310!dP,1
XXUdwhshoh2G4g,1


In [ ]:
ftp_dataset.groupby("login_password")["login_username"]\
    .nunique()\
    .sort_values(ascending=False)\
    .head(20)

,login_username
login_password,
,15
password,2
aa123456,2
admin,2
Aaaa1111,1
Aa123456,1
ASDASDASD,1
ABC123456,1
AAAAAAAA,1


In [ ]:
ftp_dataset.groupby(
    ["remote_host", "login_username", "login_password"]
).size().sort_values(ascending=False).head(20)

remote_host     login_username  login_password
35.233.105.134  anonymous       anonymous         6
35.195.54.74    anonymous       anonymous         3
35.205.86.202   anonymous       anonymous         3
165.22.70.143   sa              369369369         3
80.82.70.118    sa              19840517          2
37.8.11.194     sa              123123123a        2
                                19880511          2
74.82.47.4      sa              000000000         2
107.6.171.130   sa              19810523          2
164.52.24.165   sa              19871204          2
165.22.70.143   sa              q1w2e3r4          2
71.6.232.4      sa              19881020          2
37.8.11.194     sa              by7704566         2
165.22.70.143   sa              1q2w3e4r          2
164.52.24.165   sa              19810914          2
165.22.70.143   sa              qwertyuiop        2
37.8.11.194     sa              52113140          2
45.136.108.68   sa              19840614          1
                                19830519          1
                                19830112          1
dtype: int64

In [ ]:
ftp_dataset["hour"] = ftp_dataset["timestamp"].dt.hour

ftp_dataset["hour"].value_counts().sort_index()

,count
hour,
0,55
1,84
2,72
3,6
4,15
5,41
6,130
7,54
8,41


In [ ]:
ftp_dataset["hour"].describe()

,hour
count,1344.000000
mean,11.398065
std,6.646148
min,0.000000
25%,6.000000
50%,12.000000
75%,17.000000
max,23.000000


In [ ]:
ftp_dataset["hour"].value_counts().head(10)

,count
hour,
17,138
6,130
16,113
1,84
13,79
2,72
18,61
23,60
15,59


In [ ]:
ftp_dataset["weekday"] = ftp_dataset["timestamp"].dt.day_name()

ftp_dataset["weekday"].value_counts()

,count
weekday,
Wednesday,289
Thursday,243
Friday,199
Sunday,193
Saturday,167
Monday,152
Tuesday,101


In [ ]:
order = [
    "Monday","Tuesday","Wednesday",
    "Thursday","Friday","Saturday","Sunday"
]

ftp_dataset["weekday"] = pd.Categorical(
    ftp_dataset["weekday"],
    categories=order,
    ordered=True
)

ftp_dataset["weekday"].value_counts().sort_index()

,count
weekday,
Monday,152
Tuesday,101
Wednesday,289
Thursday,243
Friday,199
Saturday,167
Sunday,193


In [ ]:
ftp_dataset.groupby("date").size()

,0
date,
2020-01-22,123
2020-01-23,68
2020-01-24,30
2020-01-25,67
2020-01-26,51
2020-02-02,10
2020-02-03,26
2020-02-04,50
2020-02-05,19


In [ ]:
print("First attack :", ftp_dataset["timestamp"].min())
print("Last attack  :", ftp_dataset["timestamp"].max())

First attack : 2020-01-22 06:15:12.326608181
Last attack  : 2020-02-22 01:34:44.148376703


In [ ]:
ftp_dataset["connection_protocol"].value_counts(dropna=False)

,count
connection_protocol,
ftpd,1344


In [ ]:
ftp_dataset["connection_transport"].value_counts(dropna=False)

,count
connection_transport,
tcp,1344


In [ ]:
ftp_dataset["connection_type"].value_counts(dropna=False)

,count
connection_type,
accept,1344


In [ ]:
ftp_dataset["local_port"].value_counts(dropna=False)

,count
local_port,
21,1344


In [ ]:
ftp_dataset["local_host"].nunique()

1

In [ ]:
ftp_dataset["local_host"].unique()

array(['167.172.104.173'], dtype=object)

In [ ]:
ftp_dataset["remote_hostname"].value_counts(dropna=False)

,count
remote_hostname,
,1344


In [ ]:
ftp_dataset["local_port"].value_counts()

,count
local_port,
21,1344


In [ ]:
ftp_dataset["remote_port"].describe()

,remote_port
count,1344
unique,184
top,60000
freq,23


In [ ]:
ftp_dataset["connection_transport"].value_counts()

,count
connection_transport,
tcp,1344


In [ ]:
ftp_dataset.duplicated().sum()

np.int64(0)

In [ ]:
ftp_dataset.duplicated(subset=["connection"]).sum()

np.int64(1158)

In [ ]:
missing = ftp_dataset.isnull().sum()

missing[missing > 0].sort_values(ascending=False)

,0
login,9
login_username,9
login_password,9


In [ ]:
ftp_dataset["connection_protocol"].unique()

array(['ftpd'], dtype=object)

In [ ]:
ftp_dataset["connection_type"].unique()

array(['accept'], dtype=object)

In [ ]:
ftp_dataset["local_host"].nunique()

1

In [ ]:
ftp_dataset["remote_host"].nunique()

104

In [ ]:
ftp_dataset["connection"].nunique()

186

In [ ]:
len(ftp_dataset)

1344

In [ ]:
# Connections with login attempts

login_connections = ftp_dataset["login_username"].notna().sum()

print("Connections with Login Attempts:", login_connections)

print("Percentage:",
      round(login_connections / len(ftp_dataset) * 100, 2), "%")

Connections with Login Attempts: 1335
Percentage: 99.33 %


In [ ]:
print(
    "Collection Period:",
    ftp_dataset["timestamp"].max() -
    ftp_dataset["timestamp"].min()
)

Collection Period: 30 days 19:19:31.821768522


In [ ]:
print("Rows :", ftp_dataset.shape[0])
print("Columns :", ftp_dataset.shape[1])

print()

ftp_dataset.columns.tolist()

Rows : 1344
Columns : 18



['connection',
 'connection_type',
 'connection_transport',
 'connection_protocol',
 'connection_timestamp',
 'connection_root',
 'local_host',
 'local_port',
 'remote_host',
 'remote_hostname',
 'remote_port',
 'timestamp',
 'date',
 'login',
 'login_username',
 'login_password',
 'hour',
 'weekday']

In [ ]:
ftp_dataset["connection_root"].equals(
    ftp_dataset["connection"]
)

True

In [ ]:
ftp_dataset.to_csv(
    "dionaea_ftp_clean.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [ ]:
ftp_dataset.columns

Index(['connection', 'connection_type', 'connection_transport',
       'connection_protocol', 'connection_timestamp', 'connection_root',
       'local_host', 'local_port', 'remote_host', 'remote_hostname',
       'remote_port', 'timestamp', 'date', 'login', 'login_username',
       'login_password', 'hour', 'weekday'],
      dtype='object')

In [ ]:
ftp_dataset["connection_root"].nunique()

186

In [ ]:
ftp_dataset["connection"].nunique()

186

In [ ]:
ftp_dataset["connection_root"].equals(ftp_dataset["connection"])

True

In [ ]:
ftp_dataset = ftp_dataset.drop(
    columns=["connection_root"]
)

In [ ]:
ftp_dataset.columns

Index(['connection', 'connection_type', 'connection_transport',
       'connection_protocol', 'connection_timestamp', 'local_host',
       'local_port', 'remote_host', 'remote_hostname', 'remote_port',
       'timestamp', 'date', 'login', 'login_username', 'login_password',
       'hour', 'weekday'],
      dtype='object')

In [ ]:
ftp_dataset["remote_hostname"].isna().sum()

np.int64(0)

In [ ]:
ftp_dataset.drop(
columns=["remote_hostname"],
inplace=True
)

In [ ]:
ftp_dataset.columns.tolist()

['connection',
 'connection_type',
 'connection_transport',
 'connection_protocol',
 'connection_timestamp',
 'local_host',
 'local_port',
 'remote_host',
 'remote_port',
 'timestamp',
 'date',
 'login',
 'login_username',
 'login_password',
 'hour',
 'weekday']

In [ ]:
ftp_clean = ftp_dataset.drop(
    columns=[
        "connection_type",
        "connection_transport",
        "local_host"
    ]
)

ftp_clean.shape

(1344, 13)

In [ ]:
ftp_clean.isnull().sum()

,0
connection,0
connection_protocol,0
connection_timestamp,0
local_port,0
remote_host,0
remote_port,0
timestamp,0
date,0
login,9
login_username,9


In [ ]:
ftp_clean.duplicated().sum()

np.int64(0)

In [ ]:
ftp_clean.to_csv("dionaea_ftp_clean.csv", index=False)